In [1]:
# import libraries
import os
import pandas as pd
import glob

# define file paths
NEW_LAB_PATH = './model/data/lab/'
OLD_LAB_PATH = './presumed_infection/2.Data/Lab_Test_Results/'
WARD_PATH = './model/data/ward/'
OUTPUT_PATH = './model/output/'
SIC_DEMO_PATH = './presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv'

In [ ]:
# extract 2022 demographic data
df = pd.read_csv('./presumed_infection/2.Data/Demographics/demographic_SIC_all.csv')
df[df['Year'] == 2022].to_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv', index=False)

C:\Users\Vincent Yeung\AppData\Local\Temp\ipykernel_12748\2233908719.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./presumed_infection/2.Data/Demographics/demographic_SIC_all.csv')


In [ ]:
# save demographic variables
df = pd.read_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv')
SIC_data = df[['Reference Key', 'Sex', 'Admission Age (Year) (episode based)']].copy() # sex, age
SIC_data = SIC_data.drop_duplicates(subset='Reference Key') # remove dupes
SIC_data.rename(columns={'Admission Age (Year) (episode based)': 'Age'}, inplace=True)

HTN_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('hypertension', case=False, na=False)]['Reference Key'].unique() # hypertension, included all kinds
SIC_data['HTN'] = SIC_data['Reference Key'].isin(HTN_keys)

CKD_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('chronic kidney disease', case=False, na=False)]['Reference Key'].unique() # chronic kidney disease
SIC_data['CKD'] = SIC_data['Reference Key'].isin(CKD_keys)

CA_pattern = 'cancer|malign|tumor|tumour|carcinoma|sarcoma|leukemia|lymphoma|neoplasm|adenocarcinoma|melanoma' # cancer
CA_df = df[df['Dx/Px Description (HAMDCT)'].str.contains(CA_pattern, case=False, na=False)]
CA_df = CA_df[~CA_df['Dx/Px Description (HAMDCT)'].str.contains('benign', case=False, na=False)] # drop benign tumors
CA_keys = CA_df['Reference Key'].unique()
SIC_data['CA'] = SIC_data['Reference Key'].isin(CA_keys)

DM_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('diabetes mellitus', case=False, na=False)]['Reference Key'].unique() # diabetes mellitus, excluded neuropathy
SIC_data['DM'] = SIC_data['Reference Key'].isin(DM_keys)

HLD_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('hyperlipidemia', case=False, na=False)]['Reference Key'].unique() # hyperlipidemia
SIC_data['HLD'] = SIC_data['Reference Key'].isin(HLD_keys)

HF_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('heart failure|CHF', case=False, na=False)]['Reference Key'].unique() # heart failure
SIC_data['HF'] = SIC_data['Reference Key'].isin(HF_keys)

IHD_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('IHD', case=False, na=False)]['Reference Key'].unique() # ischemic heart disease
SIC_data['IHD'] = SIC_data['Reference Key'].isin(IHD_keys)

COPD_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('COPD', case=False, na=False)]['Reference Key'].unique() # COPD
SIC_data['COPD'] = SIC_data['Reference Key'].isin(COPD_keys)

Vent_keys = df[df['Dx/Px Description (HAMDCT)'].str.contains('ventilation', case=False, na=False)]['Reference Key'].unique() # ventilation
SIC_data['Vent'] = SIC_data['Reference Key'].isin(Vent_keys)

SIC_data.to_csv('./presumed_infection/2.Data/Demographics/SIC_data.csv', index=False)
SIC_data.to_csv('./model/data/SIC_data.csv', index=False)

C:\Users\Vincent Yeung\AppData\Local\Temp\ipykernel_12748\3014192335.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv')


In [ ]:
# save lab data (first 24 hours)
lab_df = pd.read_csv('./model/data/lab/8668514_sepsis_2022_P1_SIC_KEC_(Lab)_F.csv')
demo_df = pd.read_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv')
demo_df['Admission Date (yyyy-mm-dd)'] = pd.to_datetime(demo_df['Admission Date (yyyy-mm-dd)'], errors='coerce')
lab_df['LIS Reference Datetime'] = pd.to_datetime(lab_df['LIS Reference Datetime'], errors='coerce')
SIC_data = pd.read_csv('./presumed_infection/2.Data/Demographics/SIC_data.csv')

Lym_data = [] # lymphocyte (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        lym_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Lymphocyte', case=False, na=False)]['LIS Result Value']
        if len(lym_values) > 0:
            lym_min = lym_values.min()
        else:
            lym_min = None
    else:
        lym_min = None
    Lym_data.append(lym_min)
SIC_data['Lym'] = Lym_data

HB_data = [] # Hb (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        hb_values = patient_labs[patient_labs['LIS Test Description'].str.contains('HGB', case=False, na=False)]['LIS Result Value']
        if len(hb_values) > 0:
            hb_min = hb_values.min()
        else:
            hb_min = None
    else:
        hb_min = None
    HB_data.append(hb_min)

plt_df = pd.read_csv(OLD_LAB_PATH + 'Platelet/2022/plt_2022.csv')
PLT_data = [] # Platelet (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_plts = plt_df[plt_df['Reference Key'] == ref_key]
        patient_plts = patient_plts[patient_plts['LIS Reference Datetime'] >= admission]
        patient_plts = patient_plts[patient_plts['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        plt_values = patient_plts[patient_plts['LIS Test Description'].str.contains('Platelet|PLT', case=False, na=False)]['LIS Result Value']
        if len(plt_values) > 0:
            plt_min = plt_values.min()
        else:
            plt_min = None
    else:
        plt_min = None
    PLT_data.append(plt_min)
SIC_data['PLT'] = PLT_data

RDW_data = [] # RDW (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        rdw_values = patient_labs[patient_labs['LIS Test Description'].str.contains('RDW', case=False, na=False)]['LIS Result Value']
        if len(rdw_values) > 0:
            rdw_max = rdw_values.max()
        else:
            rdw_max = None
    else:
        rdw_max = None
    RDW_data.append(rdw_max)
SIC_data['RDW'] = RDW_data

K_data = [] # Potassium (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        k_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Potassium', case=False, na=False)]['LIS Result Value']
        if len(k_values) > 0:
            k_max = k_values.max()
        else:
            k_max = None
    else:
        k_max = None
    K_data.append(k_max)
SIC_data['K'] = K_data

Na_data = [] # Sodium (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        na_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Sodium', case=False, na=False)]['LIS Result Value']
        if len(na_values) > 0:
            na_min = na_values.min()
        else:
            na_min = None
    else:
        na_min = None
    Na_data.append(na_min)
SIC_data['Na'] = Na_data

Mg_data = [] # Magnesium (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        mg_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Magnesium', case=False, na=False)]['LIS Result Value']
        if len(mg_values) > 0:
            mg_max = mg_values.max()
        else:
            mg_max = None
    else:
        mg_max = None
    Mg_data.append(mg_max)
SIC_data['Mg'] = Mg_data

Ca_data = [] # Calcium (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        ca_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Calcium', case=False, na=False)]['LIS Result Value']
        if len(ca_values) > 0:
            ca_min = ca_values.min()
        else:
            ca_min = None
    else:
        ca_min = None
    Ca_data.append(ca_min)
SIC_data['Ca'] = Ca_data

Glu_data = [] # Glucose (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        glu_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Glu', case=False, na=False)]['LIS Result Value']
        if len(glu_values) > 0:
            glu_max = glu_values.max()
        else:
            glu_max = None
    else:
        glu_max = None
    Glu_data.append(glu_max)
SIC_data['Glu'] = Glu_data

Alb_data = [] # Albumin (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        alb_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Albumin', case=False, na=False)]['LIS Result Value']
        if len(alb_values) > 0:
            alb_min = alb_values.min()
        else:
            alb_min = None
    else:
        alb_min = None
    Alb_data.append(alb_min)
SIC_data['Alb'] = Alb_data

TC_data = [] # Total Cholesterol (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        tc_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Cholesterol', case=False, na=False)]['LIS Result Value']
        if len(tc_values) > 0:
            tc_max = tc_values.max()
        else:
            tc_max = None
    else:
        tc_max = None
    TC_data.append(tc_max)
SIC_data['TC'] = TC_data

lac_df = pd.read_csv(OLD_LAB_PATH + 'Lactate/2022/lac_2022.csv')
Lac_data = [] # Lactate (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_lacs = lac_df[lac_df['Reference Key'] == ref_key]
        patient_lacs = patient_lacs[patient_lacs['LIS Reference Datetime'] >= admission]
        patient_lacs = patient_lacs[patient_lacs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        lac_values = patient_lacs[patient_lacs['LIS Test Description'].str.contains('Lactate', case=False, na=False)]['LIS Result Value']
        if len(lac_values) > 0:
            lac_max = lac_values.max()
        else:
            lac_max = None
    else:
        lac_max = None
    Lac_data.append(lac_max)
SIC_data['Lac'] = Lac_data

pH_data = [] # pH (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        ph_values = patient_labs[patient_labs['LIS Test Description'].str.match('pH', na=False)]['LIS Result Value']
        if len(ph_values) > 0:
            ph_min = ph_values.min()
        else:
            ph_min = None
    else:
        ph_min = None
    pH_data.append(ph_min)
SIC_data['pH'] = pH_data

FIB_data = [] # Fibrinogen (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        fib_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Fibrinogen', case=False, na=False)]['LIS Result Value']
        if len(fib_values) > 0:
            fib_min = fib_values.min()
        else:
            fib_min = None
    else:
        fib_min = None
    FIB_data.append(fib_min)
SIC_data['FIB'] = FIB_data

ALT_data = [] # ALT (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        alt_values = patient_labs[patient_labs['LIS Test Description'].str.contains('ALT|Alanine Aminotransferase', case=True, na=False)]['LIS Result Value']
        if len(alt_values) > 0:
            alt_max = alt_values.max()
        else:
            alt_max = None
    else:
        alt_max = None
    ALT_data.append(alt_max)
SIC_data['ALT'] = ALT_data

AST_data = [] # AST (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        ast_values = patient_labs[patient_labs['LIS Test Description'].str.contains('AST|Aspartate Aminotransferase', case=True, na=False)]['LIS Result Value']
        if len(ast_values) > 0:
            ast_max = ast_values.max()
        else:
            ast_max = None
    else:
        ast_max = None
    AST_data.append(ast_max)
SIC_data['AST'] = AST_data

DBil_data = [] # Direct Bilirubin (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        dbil_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Bilirubin, Direct', case=False, na=False)]['LIS Result Value']
        if len(dbil_values) > 0:
            dbil_max = dbil_values.max()
        else:
            dbil_max = None
    else:
        dbil_max = None
    DBil_data.append(dbil_max)
SIC_data['DBil'] = DBil_data

bili_df = pd.read_csv(OLD_LAB_PATH + 'Bilirubin/2022/bili_2022.csv')
TBil_data = [] # Total Bilirubin (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_bilis = bili_df[bili_df['Reference Key'] == ref_key]
        patient_bilis = patient_bilis[patient_bilis['LIS Reference Datetime'] >= admission]
        patient_bilis = patient_bilis[patient_bilis['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        tbil_values = patient_bilis[patient_bilis['LIS Test Description'].str.contains('Bilirubin', case=False, na=False)]['LIS Result Value']
        if len(tbil_values) > 0:
            tbil_max = tbil_values.max()
        else:
            tbil_max = None
    else:
        tbil_max = None
    TBil_data.append(tbil_max)
SIC_data['TBil'] = TBil_data

UA_data = [] # Uric Acid (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        ua_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Urate', case=False, na=False)]['LIS Result Value']
        if len(ua_values) > 0:
            ua_max = ua_values.max()
        else:
            ua_max = None
    else:
        ua_max = None
    UA_data.append(ua_max)
SIC_data['UA'] = UA_data

crea_df = pd.read_csv(OLD_LAB_PATH + 'Creatinine/2022/crea_2022.csv')
Crea_data = [] # Creatinine (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_creas = crea_df[crea_df['Reference Key'] == ref_key]
        patient_creas = patient_creas[patient_creas['LIS Reference Datetime'] >= admission]
        patient_creas = patient_creas[patient_creas['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        crea_values = patient_creas[patient_creas['LIS Test Description'].str.contains('Creatinine', case=False, na=False)]['LIS Result Value']
        if len(crea_values) > 0:
            crea_max = crea_values.max()
        else:
            crea_max = None
    else:
        crea_max = None
    Crea_data.append(crea_max)
SIC_data['Crea'] = Crea_data

HCO3_data = [] # Bicarbonate (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        hco3_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Bicarbonate', case=False, na=False)]['LIS Result Value']
        if len(hco3_values) > 0:
            hco3_min = hco3_values.min()
        else:
            hco3_min = None
    else:
        hco3_min = None
    HCO3_data.append(hco3_min)
SIC_data['HCO3'] = HCO3_data

PO4_data = [] # Phosphate (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        po4_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Phosphate', case=False, na=False)]['LIS Result Value']
        if len(po4_values) > 0:
            po4_max = po4_values.max()
        else:
            po4_max = None
    else:
        po4_max = None
    PO4_data.append(po4_max)
SIC_data['PO4'] = PO4_data

BE_data = [] # Base Excess (lowest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        be_values = patient_labs[patient_labs['LIS Test Description'].str.contains('Base Excess', case=False, na=False)]['LIS Result Value']
        if len(be_values) > 0:
            be_min = be_values.min()
        else:
            be_min = None
    else:
        be_min = None
    BE_data.append(be_min)
SIC_data['BE'] = BE_data

APTT_data = [] # APTT (highest)
for _, row in SIC_data.iterrows():
    ref_key = row['Reference Key']
    admission_date = demo_df[demo_df['Reference Key'] == ref_key]['Admission Date (yyyy-mm-dd)'].values
    if len(admission_date) > 0 and pd.notna(admission_date[0]):
        admission = admission_date[0]
        patient_labs = lab_df[lab_df['Reference Key'] == ref_key]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] >= admission]
        patient_labs = patient_labs[patient_labs['LIS Reference Datetime'] <= admission + pd.Timedelta(hours=24)]
        aptt_values = patient_labs[patient_labs['LIS Test Description'].str.contains('APTT', case=False, na=False)]['LIS Result Value']
        if len(aptt_values) > 0:
            aptt_max = aptt_values.max()
        else:
            aptt_max = None
    else:
        aptt_max = None
    APTT_data.append(aptt_max)
SIC_data['APTT'] = APTT_data

SIC_data.to_csv('./presumed_infection/2.Data/Demographics/SIC_data.csv', index=False)

In [ ]:
# check ICU death within 28d
demo_df = pd.read_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv')
ward_df = pd.read_csv('./mode/data/ward/ward.csv')
ward_df['Movement Time'] = pd.to_datetime(ward_df['Movement Time'])
demo_df['Date of Registered Death'] = pd.to_datetime(demo_df['Date of Registered Death'], errors='coerce')
demo_df['ICU_death'] = False
mortality_mask = demo_df['28_day_mortality'] == True # within 28d
    
for idx, row in demo_df[mortality_mask].iterrows():
    death_date = row['Date of Registered Death']
    ref_key = row['Reference Key']
    if pd.notna(death_date):
        patient_wards = ward_df[ward_df['Reference Key'] == ref_key].sort_values('Movement Time')
        icu_pattern = 'icu|intensive care|critical care|micu|sicu|cicu|picu|nicu'
        icu_mask = patient_wards['Hospitalized Ward Name'].str.contains(icu_pattern, case=False, na=False)
        icu_periods = []
        in_icu = False
        icu_start = None
        for _, ward_row in patient_wards.iterrows():
            ward_name = str(ward_row['Hospitalized Ward Name']) if pd.notna(ward_row['Hospitalized Ward Name']) else ''
            movement_type = ward_row['Movement Type']
            movement_time = ward_row['Movement Time']
            is_icu = bool(pd.Series([ward_name]).str.contains(icu_pattern, case=False, na=False).iloc[0])
            if movement_type == 'A' and is_icu:
                in_icu = True
                icu_start = movement_time
            elif movement_type == 'D' and in_icu:
                in_icu = False
                icu_periods.append((icu_start, movement_time))
            elif movement_type == 'T' and is_icu and not in_icu:
                in_icu = True
                icu_start = movement_time
            elif movement_type == 'O' and in_icu:
                in_icu = False
                icu_periods.append((icu_start, movement_time))
            elif movement_type == 'T' and not is_icu and in_icu:
                in_icu = False
                icu_periods.append((icu_start, movement_time))
        for start, end in icu_periods:
            if start <= death_date <= end:
                demo_df.at[idx, 'ICU_death'] = True
                break

demo_df.to_csv('./presumed_infection/2.Data/Demographics/2022/demographic_SIC_2022.csv', index=False)